# Markets — classic Flink upstream + inline agentic downstream

Single Flink job: enrich → top-N → best-quote ⨝ trade → windowed features → **inline
ScreeningPipeline** (band-pass on spread + rolling z-score on spread/volumes + Claude).

Two domain mains share the same operator graph — `BondMarketAgentExample` (anonymised bond
topics fed by Python synthesizers) and `CryptoMarketAgentExample` (live Coinbase WebSocket).

This notebook stays in-JVM and drives the *agentic operator's logic* directly over a
deterministic stream of `MarketFeatures`, so you can see the funnel (RULES → ML → LLM)
without standing up Kafka. The streaming jobs run via `flink run`. Set `ANTHROPIC_API_KEY` to
light up the LLM tier here.

In [1]:
import os, pathlib, subprocess
from collections import Counter
repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = [j for j in sorted((repo_root/'target').glob('agentic-flink-*.jar'))
            if 'original' not in j.name and 'sources' not in j.name]
    assert jars, 'build the jar first: mvn -DskipTests package'
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])
cp = repo_root/'target'/'runtime-classpath.txt'
if not cp.exists():
    subprocess.run(['mvn','-q','dependency:build-classpath',f'-Dmdep.outputFile={cp}',
                    '-Dmdep.includeScope=runtime'], cwd=repo_root, check=True)
import sys
from dotenv import load_dotenv  # required — pip install python-dotenv
_env_path = repo_root / '.env'
_loaded = load_dotenv(_env_path, override=True)
print(f'.env loaded={_loaded} from {_env_path} (exists={_env_path.exists()})')
_py_src = str(repo_root / 'python')
if _py_src not in sys.path:
    sys.path.insert(0, _py_src)
import agentic_flink as af
af.start_jvm(extra_jars=[j for j in cp.read_text().strip().split(':') if j])
from agentic_flink import jclass
ScreeningPipeline = jclass('org.agentic.flink.screening.ScreeningPipeline')
ScreenItem = jclass('org.agentic.flink.screening.ScreenItem')
BandPass = jclass('org.agentic.flink.screening.BandPassDetector')
ZScore = jclass('org.agentic.flink.screening.ZScoreDetector')
Phase = jclass('org.agentic.flink.screening.Phase')
from java.util import HashMap
key = os.environ.get('ANTHROPIC_API_KEY')
print('JVM ready —', 'LLM tier: Claude' if key else 'LLM tier disabled (no ANTHROPIC_API_KEY)')

JVM ready — LLM tier disabled (no ANTHROPIC_API_KEY)


## Build the inline agentic operator (`MarketAgentFn` core, in-JVM)

These detectors are exactly what `MarketAgentFn` instantiates inside the Flink job. We feed it
a synthetic stream of `(instrument, spread, bidVolume, offerVolume)` snapshots and read the
verdict per window.

In [2]:
b = (ScreeningPipeline.builder()
       .addDetector(BandPass(0.0, 50.0, 0.6))                                    # band-pass on bid-offer spread (bp)
       .addDetector(ZScore('z-spread', Phase.CLASSIFIER, 3.0, 5, 0.7))           # spread z-score
       .addDetector(ZScore.onAttr('totalBidVolume', Phase.CLASSIFIER, 3.0, 5, 0.5))
       .addDetector(ZScore.onAttr('totalOfferVolume', Phase.CLASSIFIER, 3.0, 5, 0.5))
       .withReviewThreshold(0.6).withBlockThreshold(2.0))
if key: b = b.withClaude(key)
pipeline = b.build()
print('agentic operator built — same detectors as MarketAgentFn')

agentic operator built — same detectors as MarketAgentFn


## Stream a synthetic windowed-feature timeline and watch the funnel

In [4]:
# (instrument, spread_bp, bidVol, offerVol, t)
feed = [
    ('BND-A', 12.0, 5000, 4800, 0),
    ('BND-A', 14.0, 5200, 5100, 1000),
    ('BND-A', 13.5, 5100, 4900, 2000),
    ('BND-A', 15.0, 4900, 5050, 3000),
    ('BND-A', 14.0, 5000, 5000, 4000),
    ('BND-A', 13.0, 5050, 4950, 5000),   # baseline complete
    ('BND-A', 70.0, 5100, 4900, 6000),   # spread out-of-band + z spike
    ('BND-B', 28.0, 800,  900,  6500),   # only 1 sample on B
    ('BND-A', 14.0, 200,  4900, 7000),   # bid-volume collapse → z on attr
    ('BND-A', 13.5, 5000, 4900, 8000),
]
funnel = Counter()
for inst, spread, bv, ov, t in feed:
    attrs = HashMap()
    attrs.put('totalBidVolume', str(bv)); attrs.put('totalOfferVolume', str(ov))
    item = ScreenItem(inst, float(spread), 'spread_bp', t, attrs)
    r = pipeline.screen(item)
    funnel[str(r.decidedBy)] += 1
    phases = ','.join(sorted({str(s.phase()) for s in r.fired})) or '-'
    print(f'{str(r.decidedBy):<5} {r.verdict:<6} risk={r.combinedRisk:.2f} [{phases:<24}] | {inst} spread={spread} bid={bv} offer={ov}')
print('\nFunnel:', dict(funnel))

RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=12.0 bid=5000 offer=4800
RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=14.0 bid=5200 offer=5100
RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=13.5 bid=5100 offer=4900
RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=15.0 bid=4900 offer=5050
RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=14.0 bid=5000 offer=5000
RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=13.0 bid=5050 offer=4950
RULES REVIEW risk=1.30 [BAND_PASS,CLASSIFIER    ] | BND-A spread=70.0 bid=5100 offer=4900
RULES ALLOW  risk=0.00 [-                       ] | BND-B spread=28.0 bid=800 offer=900
RULES ALLOW  risk=0.50 [CLASSIFIER              ] | BND-A spread=14.0 bid=200 offer=4900
RULES ALLOW  risk=0.00 [-                       ] | BND-A spread=13.5 bid=5000 offer=4900

Funnel: {'RULES': 10}


## What this maps to in the Flink job

`MarketAgentFn` builds the identical `ScreeningPipeline` in its `open()` and is keyed by
`instrumentId` — so the rolling per-key window seen here matches what each Flink subtask sees in
production. The upstream operators (`EnrichmentFn`, `TopNRankerFn`, `BestQuoteFn`,
`FeatureAggregatorFn`) produce the `MarketFeatures` records this notebook simulates. Run the
full job via the launchers in `examples-bin/run-bond-market.sh` / `run-crypto-market.sh`.